### Structured output
Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

### Pydantic
Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [2]:
from langchain.chat_models import init_chat_model
model = init_chat_model(model="granite4.1:3b", model_provider="ollama")
model

ChatOllama(output_version=None, model='granite4.1:3b')

In [4]:
from pydantic import BaseModel, Field
class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="The year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The rating of the movie out of 10")

In [5]:
model_with_structured = model.with_structured_output(Movie)
model_with_structured

_ChatModelBinding(bound=ChatOllama(output_version=None, model='granite4.1:3b'), kwargs={'format': {'properties': {'title': {'description': 'The title of the movie', 'title': 'Title', 'type': 'string'}, 'year': {'description': 'The year the movie was released', 'title': 'Year', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'title': 'Director', 'type': 'string'}, 'rating': {'description': 'The rating of the movie out of 10', 'title': 'Rating', 'type': 'number'}}, 'required': ['title', 'year', 'director', 'rating'], 'title': 'Movie', 'type': 'object'}, 'ls_structured_output_format': {'kwargs': {'method': 'json_schema'}, 'schema': <class '__main__.Movie'>}}, config={}, config_factories=[])
| PydanticOutputParser(pydantic_object=<class '__main__.Movie'>)

In [14]:
model.invoke("Provide details about the movie inception")

AIMessage(content='**Inception (2010)** is a science fiction action film written and directed by Christopher Nolan. The movie explores themes of dreams, subconscious, and manipulation of time. Here are some detailed aspects of the film:\n\n### Plot Summary\nThe story follows Dom Cobb (played by Leonardo DiCaprio), a professional "extractor," someone skilled in entering people\'s dreams to steal secrets from their subconscious. However, extracting information is dangerous because entering a dream too deeply can cause "sleep paralysis" where the extractor dies.\n\nCobb is offered a chance to regain his former life: if he can perform "inception"—planting an idea in the mind of another person—a team is assembled for what seems like an impossible mission. They must enter Robert Fischer\'s (Aaron Eckhart) subconscious, where dangerous obstacles and emotional baggage await them.\n\n### Cast\n- **Leonardo DiCaprio** as Dom Cobb\n- **Joseph Gordon-Levitt** as Arthur\n- **Elliot Page** (formerly

* It seems the model is not giving the correct details and changes the director's name every time we execute.

In [16]:
model_with_structured.invoke("Provide details about inception")

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

In [17]:
model_with_structure = model.with_structured_output(Movie, include_raw=True)  

response = model_with_structure.invoke("Provide details about the movie Inception")
response

{'raw': AIMessage(content='{"title": "Inception", "year": 2010, "director": "Christopher Nolan", "rating": 8.8}', additional_kwargs={}, response_metadata={'model': 'granite4.1:3b', 'created_at': '2026-05-24T21:01:22.607557Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3538026400, 'load_duration': 59112200, 'prompt_eval_count': 15, 'prompt_eval_duration': 250691500, 'eval_count': 32, 'eval_duration': 2632956300, 'logprobs': None, 'model_name': 'granite4.1:3b', 'model_provider': 'ollama'}, id='lc_run--019e5bca-d15b-78d3-9281-cf327fb93b31-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 15, 'output_tokens': 32, 'total_tokens': 47}),
 'parsed': Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8),
 'parsing_error': None}

In [18]:
class Actor(BaseModel):
    name:str=Field(description="The name of the actor")
    role:str=Field(description="The role of the actor")

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="The year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The rating of the movie out of 10")
    actors:list[Actor]=Field(description="The list of actors in the movie")
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structured = model.with_structured_output(Movie)
response = model_with_structured.invoke("Provide details about the movie Inception")
response

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8, actors=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Ellen Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Cillian Murphy', role='Robert Fischer'), Actor(name='Marion Cotillard', role='Mal Cobb')], genres=['Science Fiction', 'Thriller', 'Action'], budget=2650000000.0)

### TypedDict
TypedDict provides a simpler alternative using Python’s built-in typing, ideal when you don’t need runtime validation.

In [20]:
from typing_extensions import Annotated, TypedDict
class Movie(TypedDict):
    title: Annotated[str, "The title of the movie"]
    year: Annotated[int, "The year the movie was released"]
    director: Annotated[str, "The director of the movie"]
    rating: Annotated[float, "The rating of the movie out of 10"]

model_with_structured = model.with_structured_output(Movie)
response = model_with_structured.invoke("Provide details about the movie Inception")
response

{'title': 'Inception',
 'year': 2010,
 'director': 'Christopher Nolan',
 'rating': 8.8}

In [24]:
print(model.profile)

None


### DataClasses
A data class is a class typically containing mainly data, although there aren’t really any restrictions. You create it using the @dataclass decorator

In [28]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent


class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model="ollama:granite4.1:3b",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='67fd806b-ffbb-455a-8d8b-38ad64958e40'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'granite4.1:3b', 'created_at': '2026-05-24T21:19:53.9771905Z', 'done': True, 'done_reason': 'stop', 'total_duration': 11108759000, 'load_duration': 1421348700, 'prompt_eval_count': 226, 'prompt_eval_duration': 6210036100, 'eval_count': 41, 'eval_duration': 3196747200, 'logprobs': None, 'model_name': 'granite4.1:3b', 'model_provider': 'ollama'}, id='lc_run--019e5bdb-a913-7f31-b5d5-2c0222add9a0-0', tool_calls=[{'name': 'ContactInfo', 'args': {'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}, 'id': '4ecb5606-b061-4da8-85fa-4bb5a18aee5e', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 226, 'output_tokens': 41, 'total_tokens': 267}),
  ToolMessage(content="Returning st

In [29]:
## Typedict
from typing_extensions import TypedDict
from langchain.agents import create_agent


class ContactInfo(TypedDict):
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person

agent = create_agent(
    model="ollama:granite4.1:3b",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]
# {'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

In [30]:
## Dataclass

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person


agent = create_agent(
    model="ollama:granite4.1:3b",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')